In [1]:
import pandas as pd
import os
import subprocess

In [2]:
# wza_kendalltau_results_bio1.csv'
# wza_results_lfmm_bio1.csv

In [3]:
tair = pd.read_csv('../key_files/TAIR10_GFF3_genes_transposons_formatted4topr_w_blocks.csv')

In [4]:
import pickle
dict_blocks = '../key_files/blocks_snpsid_dict.pkl'

with open(dict_blocks, 'rb') as file:
    dict_blocks = pickle.load(file)

reverse_mapping = {item: key for key, values in dict_blocks.items() for item in values}

In [5]:
biovar = 'bio18'

In [6]:
full_list = pd.read_csv('genes_info_BH_tair10_bio18.csv')

In [7]:
full_list = full_list[['block_id', 'model', 'gen']]

In [8]:
full_list.columns = ['block', 'model', 'gen']

In [88]:
sign_blocks_union_first_last_gen = pd.read_csv(f'top_hits_first_last_gen_{biovar}.csv')
#sign_blocks_union_first_last_gen = pd.read_csv(f'all_intersections_gea_gwas231_last_gen_blocks.csv')

sign_blocks_union_first_last_gen = pd.read_csv(f'sign_blocks_union_last_gen_BH_{biovar}.csv')

In [89]:
#sign_blocks_union_first_last_gen.columns = ['biovar', 'comparision', 'block']

In [91]:
df = sign_blocks_union_first_last_gen.copy()

In [24]:
sign_blocks_union_first_last_gen = full_list.copy()

In [25]:
df = full_list.copy()

In [26]:
bed_file_name = f"top_hits_sign_blocks_union_first_last_gen_BH_final_{biovar}.bed"

In [27]:
blocks_annotated_file = f"top_hits_sign_blocks_union_first_last_gen_blocks_annotated_BH_final_{biovar}.txt" 

In [28]:
gene_ids_file = f'top_hits_sign_blocks_union_first_last_gen_gene_ids_BH_final_{biovar}.txt'

In [29]:
gff_filtered_file = f'top_hits_sign_blocks_union_first_last_gen_BH_final_{biovar}.gff'

In [30]:
nucleotide_seq_sign_genes = f'top_hits_sign_blocks_union_first_last_gen_nucleotide_seq_BH_final_{biovar}.fa'

In [31]:
# Open the BED file for writing
with open(bed_file_name, "w") as bed_file:
    for block in df['block']:
        if block in dict_blocks:
            # Extract SNPs for the block
            snps = dict_blocks[block]
            
            # Assuming the format of SNP is like '1_291' where '1' is the chromosome and '291' is the position
            # We need to find the min and max positions for the block to create the start-end range
            chrom_positions = [snp.split('_') for snp in snps]
            chrom = chrom_positions[0][0]  # The chromosome, assuming all SNPs are from the same chromosome
            
            # Prepend 'Chr' to the chromosome name to match GFF naming convention
            chrom = f"Chr{chrom}"
            
            # Get the minimum and maximum position for the block
            start = min(int(pos) for chrom, pos in chrom_positions)
            end = max(int(pos) for chrom, pos in chrom_positions)
            
            # Write the block information in BED format: chrom, start, end, block name
            bed_file.write(f"{chrom}\t{start}\t{end}\t{block}\n")

print("BED file has been created with 'Chr' prefix in chromosome names.")


BED file has been created with 'Chr' prefix in chromosome names.


In [32]:
bed_file_name

'top_hits_sign_blocks_union_first_last_gen_BH_final_bio18.bed'

In [33]:
## this will, based on the TAIR10_GFF3_genes_transposons cehck the intersectionin between the bed file fo hte singificant blocks and the genes 
gff_file = "../key_files/TAIR10_GFF3_genes_transposons.gff"  # Replace with your actual GFF3 file path

# Construct the command to load the bedtools module and run the intersect command
cmd = f"""
module load bedtools/2.31.1; \
bedtools intersect -wb -a {bed_file_name} -b {gff_file} > {blocks_annotated_file}
"""

# Run the command
try:
    subprocess.run(cmd, shell=True, check=True, executable="/bin/bash")  # Use bash to run the command
    print(f"Intersection complete. Results saved in {blocks_annotated_file}")
except subprocess.CalledProcessError as e:
    print(f"Error running bedtools: {e}")


Intersection complete. Results saved in top_hits_sign_blocks_union_first_last_gen_blocks_annotated_BH_final_bio18.txt


In [34]:
annot = pd.read_csv(blocks_annotated_file, sep='\t', header=None)

In [35]:
annot.columns = [
    'block_chrom', 'block_start', 'block_end', 'block_id',  # From BED file
    'gene_chrom', 'source', 'feature', 'gene_start', 'gene_end',  # From GFF file
    'score', 'strand', 'frame', 'attribute'  # From GFF file
]

In [36]:
annot = annot[annot['attribute'].str.contains('protein_coding_gene')]

In [37]:
annot['gene_id'] = annot['attribute'].str.split(';').str[0].str.replace('ID=', '')

In [38]:
sign_blocks_union_first_last_gen = sign_blocks_union_first_last_gen.merge(annot, left_on = 'block', right_on = 'block_id')

In [39]:
sign_blocks_union_first_last_gen

,block,model,gen,block_chrom,block_start,block_end,block_id,gene_chrom,source,feature,gene_start,gene_end,score,strand,frame,attribute,gene_id
0,2_841,wza_lfmm_l,last_gen,Chr2,7668077,7670495,2_841,Chr2,TAIR10,gene,7668078,7670495,.,-,.,ID=AT2G17640;Note=protein_coding_gene;Name=AT2...,AT2G17640
1,2_841,wza_lfmm_l,last_gen,Chr2,7671015,7673036,2_841,Chr2,TAIR10,gene,7671016,7673036,.,+,.,ID=AT2G17650;Note=protein_coding_gene;Name=AT2...,AT2G17650
2,2_841,wza_lfmm_l,last_gen,Chr2,7673281,7673712,2_841,Chr2,TAIR10,gene,7673282,7673712,.,+,.,ID=AT2G17660;Note=protein_coding_gene;Name=AT2...,AT2G17660
3,2_841,wza_lfmm_l,last_gen,Chr2,7674350,7676157,2_841,Chr2,TAIR10,gene,7674351,7676157,.,+,.,ID=AT2G17670;Note=protein_coding_gene;Name=AT2...,AT2G17670
4,2_841,wza_lfmm_l,last_gen,Chr2,7679007,7680378,2_841,Chr2,TAIR10,gene,7679008,7680378,.,+,.,ID=AT2G17680;Note=protein_coding_gene;Name=AT2...,AT2G17680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
981,5_733,wza_binom_reg_l,first_gen,Chr5,8677090,8682205,5_733,Chr5,TAIR10,gene,8677091,8682205,.,+,.,ID=AT5G25150;Note=protein_coding_gene;Name=AT5...,AT5G25150
982,5_733,wza_binom_reg_l,first_gen,Chr5,8687456,8688415,5_733,Chr5,TAIR10,gene,8687457,8688415,.,+,.,ID=AT5G25160;Note=protein_coding_gene;Name=AT5...,AT5G25160
983,5_733,wza_binom_reg_l,first_gen,Chr5,8693136,8695095,5_733,Chr5,TAIR10,gene,8693137,8695095,.,+,.,ID=AT5G25170;Note=protein_coding_gene;Name=AT5...,AT5G25170
984,5_733,wza_binom_reg_l,first_gen,Chr5,8694629,8697108,5_733,Chr5,TAIR10,gene,8694630,8697108,.,-,.,ID=AT5G25180;Note=protein_coding_gene;Name=AT5...,AT5G25180


In [40]:
sign_blocks_union_first_last_gen.to_csv(f'top_hits_gene_and_model_sign_blocks_union_first_last_gen_BH_{biovar}.csv')

In [41]:
# Assuming your DataFrame has a column for gene IDs, e.g., 'gene_id'
gene_ids = annot['gene_id'].unique()  # Get unique gene IDs

# Write these gene IDs to a file to use for filtering the GFF3 file
with open(gene_ids_file, 'w') as f:
    for gene_id in gene_ids:
        f.write(f"{gene_id}\n")
        
print("Gene IDs saved to gene_ids.txt")

Gene IDs saved to gene_ids.txt


In [42]:
gene_ids = pd.read_csv(gene_ids_file)

In [43]:
gene_ids

,AT2G17640
0,AT2G17650
1,AT2G17660
2,AT2G17670
3,AT2G17680
4,AT2G17690
...,...
100,AT5G25150
101,AT5G25160
102,AT5G25170
103,AT5G25180


In [72]:
# Step 2: Filter the GFF3 file using the list of gene IDs
filter_cmd = f"grep -Ff {gene_ids_file} ../key_files/TAIR10_GFF3_genes_transposons.gff > {gff_filtered_file}"
subprocess.run(filter_cmd, shell=True, check=True)

CompletedProcess(args='grep -Ff top_hits_sign_blocks_union_first_last_gen_gene_ids_BH_final_bio18.txt ../key_files/TAIR10_GFF3_genes_transposons.gff > top_hits_sign_blocks_union_first_last_gen_BH_final_bio18.gff', returncode=0)

In [73]:
# Define a function to replace chromosome names in the GFF file
def replace_chromosomes(gff_file):
    # Read the GFF file
    with open(gff_file, 'r') as file:
        gff_content = file.read()
    
    # Replace Chr1, Chr2, ..., with 1, 2, ..., and leave ChrC and ChrM unchanged
    gff_content = gff_content.replace('Chr1', '1')
    gff_content = gff_content.replace('Chr2', '2')
    gff_content = gff_content.replace('Chr3', '3')
    gff_content = gff_content.replace('Chr4', '4')
    gff_content = gff_content.replace('Chr5', '5')
    # ChrC and ChrM remain unchanged, no need for replacements
    
    # Write the modified content back to the GFF file
    with open(gff_file, 'w') as file:
        file.write(gff_content)

# Specify your GFF file
gff_file = gff_filtered_file

# Call the function to replace chromosome names
replace_chromosomes(gff_file)

print("Chromosome names replaced successfully.")


Chromosome names replaced successfully.


In [74]:
# Step 3: Use gffread to extract sequences
gffread_cmd = f"""
module load GffRead/0.12.3; \
gffread -w {nucleotide_seq_sign_genes} -g ../key_files/arabidopsis_reference/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa {gff_file}
"""

subprocess.run(gffread_cmd, shell=True, check=True, executable="/bin/bash")
print("Nucleotide sequences saved to output_transcripts.fa")

Lmod has detected the following error: The following module(s) are unknown:
"GffRead/0.12.3"

Please check the spelling or version number. Also try "module spider ..."
It is also possible your cache file is out-of-date; it may help to try:
  $ module --ignore_cache load "GffRead/0.12.3"

Also make sure that all modulefiles written in TCL start with the string
#%Module



/bin/bash: line 2: gffread: command not found


CalledProcessError: Command '
module load GffRead/0.12.3; gffread -w top_hits_sign_blocks_union_first_last_gen_nucleotide_seq_BH_final_bio18.fa -g ../key_files/arabidopsis_reference/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa top_hits_sign_blocks_union_first_last_gen_BH_final_bio18.gff
' returned non-zero exit status 127.

In [79]:
# conda activate /home/tbellagio/miniforge3/envs/diamond_env

In [37]:
biovar

'bio18'

In [44]:

false_disc = pd.read_csv(f'top_hits_gene_and_model_sign_blocks_union_first_last_gen_BH_{biovar}.csv')

In [45]:
false_disc

,Unnamed: 0,block,model,gen,block_chrom,block_start,block_end,block_id,gene_chrom,source,feature,gene_start,gene_end,score,strand,frame,attribute,gene_id
0,0,2_841,wza_lfmm_l,last_gen,Chr2,7668077,7670495,2_841,Chr2,TAIR10,gene,7668078,7670495,.,-,.,ID=AT2G17640;Note=protein_coding_gene;Name=AT2...,AT2G17640
1,1,2_841,wza_lfmm_l,last_gen,Chr2,7671015,7673036,2_841,Chr2,TAIR10,gene,7671016,7673036,.,+,.,ID=AT2G17650;Note=protein_coding_gene;Name=AT2...,AT2G17650
2,2,2_841,wza_lfmm_l,last_gen,Chr2,7673281,7673712,2_841,Chr2,TAIR10,gene,7673282,7673712,.,+,.,ID=AT2G17660;Note=protein_coding_gene;Name=AT2...,AT2G17660
3,3,2_841,wza_lfmm_l,last_gen,Chr2,7674350,7676157,2_841,Chr2,TAIR10,gene,7674351,7676157,.,+,.,ID=AT2G17670;Note=protein_coding_gene;Name=AT2...,AT2G17670
4,4,2_841,wza_lfmm_l,last_gen,Chr2,7679007,7680378,2_841,Chr2,TAIR10,gene,7679008,7680378,.,+,.,ID=AT2G17680;Note=protein_coding_gene;Name=AT2...,AT2G17680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
981,981,5_733,wza_binom_reg_l,first_gen,Chr5,8677090,8682205,5_733,Chr5,TAIR10,gene,8677091,8682205,.,+,.,ID=AT5G25150;Note=protein_coding_gene;Name=AT5...,AT5G25150
982,982,5_733,wza_binom_reg_l,first_gen,Chr5,8687456,8688415,5_733,Chr5,TAIR10,gene,8687457,8688415,.,+,.,ID=AT5G25160;Note=protein_coding_gene;Name=AT5...,AT5G25160
983,983,5_733,wza_binom_reg_l,first_gen,Chr5,8693136,8695095,5_733,Chr5,TAIR10,gene,8693137,8695095,.,+,.,ID=AT5G25170;Note=protein_coding_gene;Name=AT5...,AT5G25170
984,984,5_733,wza_binom_reg_l,first_gen,Chr5,8694629,8697108,5_733,Chr5,TAIR10,gene,8694630,8697108,.,-,.,ID=AT5G25180;Note=protein_coding_gene;Name=AT5...,AT5G25180


In [46]:
false_disc = false_disc.drop_duplicates(subset=['gene_id'])

In [47]:
false_disc

,Unnamed: 0,block,model,gen,block_chrom,block_start,block_end,block_id,gene_chrom,source,feature,gene_start,gene_end,score,strand,frame,attribute,gene_id
0,0,2_841,wza_lfmm_l,last_gen,Chr2,7668077,7670495,2_841,Chr2,TAIR10,gene,7668078,7670495,.,-,.,ID=AT2G17640;Note=protein_coding_gene;Name=AT2...,AT2G17640
1,1,2_841,wza_lfmm_l,last_gen,Chr2,7671015,7673036,2_841,Chr2,TAIR10,gene,7671016,7673036,.,+,.,ID=AT2G17650;Note=protein_coding_gene;Name=AT2...,AT2G17650
2,2,2_841,wza_lfmm_l,last_gen,Chr2,7673281,7673712,2_841,Chr2,TAIR10,gene,7673282,7673712,.,+,.,ID=AT2G17660;Note=protein_coding_gene;Name=AT2...,AT2G17660
3,3,2_841,wza_lfmm_l,last_gen,Chr2,7674350,7676157,2_841,Chr2,TAIR10,gene,7674351,7676157,.,+,.,ID=AT2G17670;Note=protein_coding_gene;Name=AT2...,AT2G17670
4,4,2_841,wza_lfmm_l,last_gen,Chr2,7679007,7680378,2_841,Chr2,TAIR10,gene,7679008,7680378,.,+,.,ID=AT2G17680;Note=protein_coding_gene;Name=AT2...,AT2G17680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
701,701,5_733,wza_binom_reg_l,first_gen,Chr5,8677090,8682205,5_733,Chr5,TAIR10,gene,8677091,8682205,.,+,.,ID=AT5G25150;Note=protein_coding_gene;Name=AT5...,AT5G25150
702,702,5_733,wza_binom_reg_l,first_gen,Chr5,8687456,8688415,5_733,Chr5,TAIR10,gene,8687457,8688415,.,+,.,ID=AT5G25160;Note=protein_coding_gene;Name=AT5...,AT5G25160
703,703,5_733,wza_binom_reg_l,first_gen,Chr5,8693136,8695095,5_733,Chr5,TAIR10,gene,8693137,8695095,.,+,.,ID=AT5G25170;Note=protein_coding_gene;Name=AT5...,AT5G25170
704,704,5_733,wza_binom_reg_l,first_gen,Chr5,8694629,8697108,5_733,Chr5,TAIR10,gene,8694630,8697108,.,-,.,ID=AT5G25180;Note=protein_coding_gene;Name=AT5...,AT5G25180


In [48]:
false_disc = false_disc.drop('Unnamed: 0',axis=1).drop_duplicates()

In [49]:
false_disc

,block,model,gen,block_chrom,block_start,block_end,block_id,gene_chrom,source,feature,gene_start,gene_end,score,strand,frame,attribute,gene_id
0,2_841,wza_lfmm_l,last_gen,Chr2,7668077,7670495,2_841,Chr2,TAIR10,gene,7668078,7670495,.,-,.,ID=AT2G17640;Note=protein_coding_gene;Name=AT2...,AT2G17640
1,2_841,wza_lfmm_l,last_gen,Chr2,7671015,7673036,2_841,Chr2,TAIR10,gene,7671016,7673036,.,+,.,ID=AT2G17650;Note=protein_coding_gene;Name=AT2...,AT2G17650
2,2_841,wza_lfmm_l,last_gen,Chr2,7673281,7673712,2_841,Chr2,TAIR10,gene,7673282,7673712,.,+,.,ID=AT2G17660;Note=protein_coding_gene;Name=AT2...,AT2G17660
3,2_841,wza_lfmm_l,last_gen,Chr2,7674350,7676157,2_841,Chr2,TAIR10,gene,7674351,7676157,.,+,.,ID=AT2G17670;Note=protein_coding_gene;Name=AT2...,AT2G17670
4,2_841,wza_lfmm_l,last_gen,Chr2,7679007,7680378,2_841,Chr2,TAIR10,gene,7679008,7680378,.,+,.,ID=AT2G17680;Note=protein_coding_gene;Name=AT2...,AT2G17680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
701,5_733,wza_binom_reg_l,first_gen,Chr5,8677090,8682205,5_733,Chr5,TAIR10,gene,8677091,8682205,.,+,.,ID=AT5G25150;Note=protein_coding_gene;Name=AT5...,AT5G25150
702,5_733,wza_binom_reg_l,first_gen,Chr5,8687456,8688415,5_733,Chr5,TAIR10,gene,8687457,8688415,.,+,.,ID=AT5G25160;Note=protein_coding_gene;Name=AT5...,AT5G25160
703,5_733,wza_binom_reg_l,first_gen,Chr5,8693136,8695095,5_733,Chr5,TAIR10,gene,8693137,8695095,.,+,.,ID=AT5G25170;Note=protein_coding_gene;Name=AT5...,AT5G25170
704,5_733,wza_binom_reg_l,first_gen,Chr5,8694629,8697108,5_733,Chr5,TAIR10,gene,8694630,8697108,.,-,.,ID=AT5G25180;Note=protein_coding_gene;Name=AT5...,AT5G25180


In [50]:
from Bio import Entrez

# Provide your email, NCBI requires this for queries
Entrez.email = "tbellg@berkeley.edu"

# Function to retrieve gene information using NCBI Entrez API
def get_gene_info_ncbi(gene_id):
    try:
        # Search for the gene in the NCBI database
        search_handle = Entrez.esearch(db="gene", term=f"{gene_id}[Gene]", retmax=1)
        search_results = Entrez.read(search_handle)
        search_handle.close()
        
        if search_results["IdList"]:
            gene_ncbi_id = search_results["IdList"][0]  # Get the first matching gene ID
            
            # Fetch detailed gene information using the gene ID
            fetch_handle = Entrez.efetch(db="gene", id=gene_ncbi_id, retmode="xml")
            gene_data = Entrez.read(fetch_handle)
            fetch_handle.close()
            
            # Extract relevant information from the gene data
            gene_info = gene_data[0]
            gene_name = gene_info['Entrezgene_gene']['Gene-ref']['Gene-ref_locus']
            gene_desc = gene_info['Entrezgene_summary']
            return {
                "gene_id": gene_id,
                "gene_name": gene_name,
                "description": gene_desc
            }
        else:
            return None
    except Exception as e:
        print(f"Error fetching data for gene {gene_id}: {e}")
        return None

# Initialize an empty list to store gene information and the corresponding block
gene_info_list = []
    # Loop through each gene_id and retrieve information from the API

for index,row in false_disc.iterrows():
    gene_id =  row['gene_id']
    block = row['block']
    print(gene_id)
    gene_info = get_gene_info_ncbi(gene_id.strip())  # Ensure any extra spaces are removed
    if gene_info:
        # Add the gene information and the corresponding block to the list
        gene_info_list.append({
            "block": block,
            "gene_id": gene_info.get('gene_id'),
            "gene_name": gene_info.get('gene_name'),
            "description": gene_info.get('description')
        })

AT2G17640
AT2G17650
Error fetching data for gene AT2G17650: 'Gene-ref_locus'
AT2G17660
Error fetching data for gene AT2G17660: 'Gene-ref_locus'
AT2G17670
Error fetching data for gene AT2G17670: 'Gene-ref_locus'
AT2G17680
Error fetching data for gene AT2G17680: 'Gene-ref_locus'
AT2G17690
AT2G17695
Error fetching data for gene AT2G17695: 'Gene-ref_locus'
AT2G17700
Error fetching data for gene AT2G17700: 'Entrezgene_summary'
AT2G17705
Error fetching data for gene AT2G17705: 'Gene-ref_locus'
AT2G17710
Error fetching data for gene AT2G17710: 'Gene-ref_locus'
AT2G17720
AT2G17630
Error fetching data for gene AT2G17630: 'Gene-ref_locus'
AT2G17723
Error fetching data for gene AT2G17723: 'Gene-ref_locus'
AT3G29350
AT3G29360
AT5G03190
Error fetching data for gene AT5G03190: 'Entrezgene_summary'
AT5G03200
AT5G54130
Error fetching data for gene AT5G54130: 'Gene-ref_locus'
AT5G54140
AT1G23550
AT1G31740
AT2G27030
AT2G14890
AT4G13980
AT4G19230
AT4G39520
Error fetching data for gene AT4G39520: 'Gene-re

In [51]:
df_gene_info = pd.DataFrame(gene_info_list)

In [ ]:
AAE18

In [57]:
df_gene_info[df_gene_info['gene_name'] == 'AAE18']

,block,gene_id,gene_name,description
23,1_2859,AT1G55320,AAE18,Encodes a protein with similarity to acyl acti...


In [53]:
df_gene_info[df_gene_info['gene_name'].isin(gene_names)].to_csv('genes_id.csv',index=None)

NameError: name 'gene_names' is not defined

In [58]:
df_gene_info.to_csv('genes_id.csv',index=None)

In [124]:
pd.Series(gene_names).unique()

array(['ATSERAT3;1', 'SDC', 'P4H5', 'AHP2', 'UGD2', 'LUL1', 'ILL3',
       'SRO2', 'BGAL15', 'CAM5', 'AGP9', 'AT-HSFA5', 'CYP707A1', 'SK2',
       'ACS12', 'ILL6', 'RGA1', 'PRD1', 'ATCSLB05', 'HSF A4A', 'IOS1',
       'GLIP2', 'SCL33', 'AAE18', 'AGP21', 'GCT', 'DEK1', 'MAP70-1',
       'FKF1', 'GAI', 'CPN10', 'QSOX1', 'GAT1_2.1', 'EER4', 'ARA1',
       'PLDGAMMA1', 'SMO1-1', 'SEC1B', 'WNK9', 'CYSD2', 'CYP71A19',
       'TPS12', 'ABF2', 'THO5', 'NRAMP5', 'NAC089', 'CIPK25', 'CYP71B11',
       'CYP71B12', 'CYP71B13', 'TAF5', 'ZFP3', 'ESE3'], dtype=object)

In [121]:
c = [
    "ATSERAT3;1",
    "SDC",
    "P4H5",
    "AHP2",
    "UGD2",
    "LUL1",
    "LUL1",
    "LUL1",
    "LUL1",
    "ILL3",
    "SRO2",
    "BGAL15",
    "CAM5",
    "AGP9",
    "AT-HSFA5",
    "CYP707A1",
    "SK2",
    "ACS12",
    "ILL6",
    "RGA1",
    "PRD1",
    "ATCSLB05",
    "HSF A4A",
    "IOS1",
    "GLIP2",
    "SCL33",
    "AAE18",
    "AGP21",
    "GCT",
    "DEK1",
    "MAP70-1",
    "FKF1",
    "GAI",
    "GAI",
    "GAI",
    "GAI",
    "CPN10",
    "QSOX1",
    "GAT1_2.1",
    "EER4",
    "ARA1",
    "PLDGAMMA1",
    "SMO1-1",
    "SEC1B",
    "WNK9",
    "CYSD2",
    "CYP71A19",
    "TPS12",
    "ABF2",
    "THO5",
    "NRAMP5",
    "NAC089",
    "CIPK25",
    "CYP71B11",
    "CYP71B12",
    "CYP71B13",
    "TAF5",
    "ZFP3",
    "ESE3"
]


In [83]:
df_gene_info

,block,gene_id,gene_name,description
0,3_2534,AT3G47500,CDF3,Dof-type zinc finger domain-containing protein...
1,5_724,AT5G24670,EMB2820,A protein coding gene with unknown function. T...
2,5_724,AT5G24770,VSP2,Has acid phosphatase activity dependent on the...
3,5_724,AT5G24780,VSP1,encodes an acid phosphatase similar to soybean...
4,5_724,AT5G24800,BZIP9,Encodes bZIP protein BZO2H2.
...,...,...,...,...
672,5_335,AT5G09870,CESA5,Encodes a cellulose synthase CESA5 that produc...
673,5_335,AT5G09610,PUM21,Encodes a member of the Arabidopsis Pumilio (A...
674,5_335,AT5G09805,IDL3,Similar to Inflorescence deficient in abscissi...
675,5_630,AT5G23720,PHS1,Encodes a protein tyrosine phosphatase Propyza...


In [29]:
false_disc = false_disc.merge(df_gene_info, on = 'gene_id')

In [31]:
false_disc = false_disc.drop_duplicates()

In [84]:
false_disc[false_disc['gene_name'] == 'LUL1']

KeyError: 'gene_name'

In [33]:
false_disc[['gene_name', 'gene_id']].drop_duplicates()

,gene_name,gene_id
0,CDF3,AT3G47500
1,EMB2820,AT5G24670
2,VSP2,AT5G24770
3,VSP1,AT5G24780
4,BZIP9,AT5G24800
...,...,...
684,CESA5,AT5G09870
685,PUM21,AT5G09610
686,IDL3,AT5G09805
687,PHS1,AT5G23720


In [65]:
false_disc = false_disc[['biovar', 'comparision','block_id', 'block_chrom', 'block_start',
       'block_end',
       'gene_id',  'gene_name', 'description']]

In [71]:
false_disc = false_disc.drop_duplicates()

In [72]:
false_disc.to_csv('genes_overlap_methods.csv',index=None)

In [ ]:
#bonferroni.merge(df_gene_info, on = 'gene_id')[['block_id', 'model', 'gen', 'source', 'gene_name', 'description']].to_csv('genes_info_bonferronicor_tair10.csv',index=None)

In [ ]:
false_disc

In [90]:
false_disc = false_disc.drop_duplicates()

In [91]:
false_disc.to_csv(f'top_hits_genes_info_tair10_{biovar}.csv',index=None)

In [118]:
df_gene_info.to_csv('genes_info_blocks_first_last_BHcorr_pre_blast.csv')

In [77]:
flowering_gene_ids = ['AT1G65480',  # FT
                      'AT1G74930',  # FLC
                      'AT1G04810',  # SOC1
                      'AT1G69120',  # AP1
                      'AT5G02470',  # FUL
                      'AT5G01330',  # LFY
                      'AT3G10040',  # AP2
                      'AT4G02560']  # SVP

In [78]:
sign_wza_lfmm

NameError: name 'sign_wza_lfmm' is not defined

In [120]:
df_gene_info_ncbi

,block,gene_id,gene_name,description
0,1_2867,AT1G55670,PSAG,"Encodes subunit G of photosystem I, an 11-kDa ..."
1,1_485,AT1G11840,GLX1,Encodes a glyoxalase I homolog ATGLX1.
2,1_485,AT1G70580,AOAT2,Encodes a protein with glyoxylate aminotransfe...
3,1_485,AT1G70610,ABCB26,member of TAP subfamily
4,1_485,AT1G70660,MMZ2,MMZ2/UEV1B encodes a protein that may play a r...
...,...,...,...,...
59,5_2922,AT5G60220,TET4,Member of TETRASPANIN family
60,5_2922,AT5G60230,SEN2,putative subunit of tRNA splicing endonuclease
61,5_2922,AT5G60300,LecRK-I.9,Lectin Receptor Kinase involved in protein-pro...
62,5_2922,AT5G60340,AAK6,Encodes a nuclear adenylate kinase that intera...


In [107]:
df_gene_info_ncbi = df_gene_info.copy()

In [109]:
df_gene_info_ncbi.to_csv('df_lfmm_wza_block_gene_info_ncbi.csv',index=None)

In [114]:
df_gene_info_ncbi['block'].unique()

array(['1_2867', '1_485', '1_702', '2_1265', '2_827', '2_970', '3_137',
       '3_250', '4_801', '5_1947', '5_2922'], dtype=object)

In [121]:
['1_485', '1_702', '1_2867', '2_827', '2_970','2_1265', '3_137',
       '3_250', '4_801', '5_1947', '5_2922']

['1_485',
 '1_702',
 '1_2867',
 '2_827',
 '2_970',
 '2_1265',
 '3_137',
 '3_250',
 '4_801',
 '5_1947',
 '5_2922']

In [123]:
for i in ['1_485', '1_702', '1_2867', '2_827', '2_970','2_1265', '3_137',
       '3_250', '4_801', '5_1947', '5_2922']:
    print(i)
    print(dict_blocks[i][0])
    print(dict_blocks[i][-1])

1_485
1_3995653
1_3997916
1_702
1_6018870
1_6181134
1_2867
1_20800345
1_20814896
2_827
2_7553538
2_7567816
2_970
2_9540996
2_9714921
2_1265
2_11533904
2_11534263
3_137
3_1540490
3_1548153
3_250
3_2346807
3_2350128
4_801
4_7255726
4_7269426
5_1947
5_17738817
5_17759685
5_2922
5_24100585
5_24286474


In [27]:
df_gene_info_ensembl

,block,gene_id,gene_name,start,end,biotype,description
0,1_1318,AT1G26680,AT1G26680,9219460,9223795,protein_coding,transcriptional factor B3 family protein [Sour...
1,1_1594,AT1G30710,AT1G30710,10895252,10897127,protein_coding,FAD-binding Berberine family protein [Source:N...
2,1_1594,AT1G30720,AT1G30720,10897925,10899975,protein_coding,FAD-binding Berberine family protein [Source:N...
3,1_2867,AT1G55660,AT1G55660,20800694,20802377,protein_coding,"FBD, F-box and Leucine Rich Repeat domains con..."
4,1_2867,AT1G55675,AT1G55675,20803661,20804574,protein_coding,transmembrane protein [Source:NCBI gene (forme...
...,...,...,...,...,...,...,...
116,5_2922,AT5G60335,AT5G60335,24272995,24274860,protein_coding,hydroxyacyl-thioester dehydratase type-like pr...
117,5_2922,AT5G60350,AT5G60350,24277730,24279147,protein_coding,None
118,5_2922,AT5G60370,AT5G60370,24282666,24285061,protein_coding,exonuclease V-like protein [Source:NCBI gene (...
119,5_2922,AT5G60380,AT5G60380,24285920,24287808,protein_coding,"transmembrane protein, putative (DUF239) [Sour..."


In [28]:
df_gene_info_ncbi

,block,gene_id,gene_name,description
0,1_2867,AT1G55670,PSAG,"Encodes subunit G of photosystem I, an 11-kDa ..."
1,1_485,AT1G11840,GLX1,Encodes a glyoxalase I homolog ATGLX1.
2,1_485,AT1G70580,AOAT2,Encodes a protein with glyoxylate aminotransfe...
3,1_485,AT1G70610,ABCB26,member of TAP subfamily
4,1_485,AT1G70660,MMZ2,MMZ2/UEV1B encodes a protein that may play a r...
...,...,...,...,...
59,5_2922,AT5G60220,TET4,Member of TETRASPANIN family
60,5_2922,AT5G60230,SEN2,putative subunit of tRNA splicing endonuclease
61,5_2922,AT5G60300,LecRK-I.9,Lectin Receptor Kinase involved in protein-pro...
62,5_2922,AT5G60340,AAK6,Encodes a nuclear adenylate kinase that intera...


In [34]:
len(flattened_gene_ids)

207

In [44]:
tair['biotype'].unique()

array(['protein_coding_gene'], dtype=object)

In [42]:
tair['gene_id'].nunique()

27206

In [38]:
flattened_gene_ids

['AT1G26680',
 'AT1G30710',
 'AT1G30720',
 'AT1G55660',
 'AT1G55670',
 'AT1G55675',
 'AT1G55680',
 'AT1G55690',
 'AT1G67560',
 'AT1G67570',
 'AT1G70020',
 'AT1G11840',
 'AT1G70580',
 'AT1G70590',
 'AT1G70600',
 'AT1G70610',
 'AT1G70620',
 'AT1G70630',
 'AT1G70640',
 'AT1G70650',
 'AT1G70660',
 'AT1G17500',
 'AT1G17510',
 'AT1G17520',
 'AT1G17530',
 'AT1G17540',
 'AT1G17545',
 'AT1G17550',
 'AT1G17560',
 'AT1G17580',
 'AT1G17590',
 'AT1G17600',
 'AT1G17610',
 'AT1G17615',
 'AT1G17620',
 'AT1G17630',
 'AT1G17640',
 'AT1G17650',
 'AT1G17665',
 'AT1G17680',
 'AT1G17690',
 'AT1G17700',
 'AT1G17710',
 'AT1G17720',
 'AT1G17730',
 'AT1G17744',
 'AT1G17745',
 'AT1G17750',
 'AT1G17760',
 'AT1G17770',
 'AT1G17780',
 'AT1G17790',
 'AT1G17800',
 'AT1G17810',
 'AT1G17820',
 'AT1G17830',
 'AT1G17840',
 'AT1G17850',
 'AT1G17860',
 'AT1G17870',
 'AT1G17880',
 'AT1G17890',
 'AT1G17910',
 'AT1G17920',
 'AT1G17930',
 'AT1G17940',
 'AT1G17950',
 'AT1G17960',
 'AT2G24230',
 'AT2G27030',
 'AT2G17380',
 'AT2G

In [35]:
set(df_gene_info_ncbi['gene_id']).intersection(df_gene_info_ensembl['gene_id'])

set()

In [36]:
len(set(flattened_gene_ids).intersection(set(df_gene_info_ncbi['gene_id'])))

64

In [37]:
len(set(flattened_gene_ids).intersection(set(df_gene_info_ensembl['gene_id'])))

121

In [50]:
121 + 64

185

In [49]:
len(flattened_gene_ids)

207

In [28]:
df_gene_info_ensembl.merge(df_gene_info_ncbi, on = 'gene_id')

,block_x,gene_id,gene_name_x,start,end,biotype,description_x,block_y,gene_name_y,description_y


In [ ]:
df_gene_info_ensembl

In [ ]:
from Bio import Entrez

# Provide your email, NCBI requires this for queries
Entrez.email = "tbellg@berkeley.edu"

# Function to retrieve gene information using NCBI Entrez API
def get_gene_info_ncbi(gene_id):
    try:
        # Search for the gene in the NCBI database
        search_handle = Entrez.esearch(db="gene", term=f"{gene_id}[Gene]", retmax=1)
        search_results = Entrez.read(search_handle)
        search_handle.close()
        
        if search_results["IdList"]:
            gene_ncbi_id = search_results["IdList"][0]  # Get the first matching gene ID
            
            # Fetch detailed gene information using the gene ID
            fetch_handle = Entrez.efetch(db="gene", id=gene_ncbi_id, retmode="xml")
            gene_data = Entrez.read(fetch_handle)
            fetch_handle.close()
            
            # Extract relevant information from the gene data
            gene_info = gene_data[0]
            gene_name = gene_info['Entrezgene_gene']['Gene-ref']['Gene-ref_locus']
            gene_desc = gene_info['Entrezgene_summary']
            return {
                "gene_id": gene_id,
                "gene_name": gene_name,
                "description": gene_desc
            }
        else:
            return None
    except Exception as e:
        print(f"Error fetching data for gene {gene_id}: {e}")
        return None


In [36]:
sign_wza_lfmm['matching_gene_id'].values

array(['AT1G26680', 'AT1G30710,AT1G30720',
       'AT1G55660,AT1G55670,AT1G55675,AT1G55680,AT1G55690',
       'AT1G67560,AT1G67570', 'AT1G70020',
       'AT1G11840,AT1G70580,AT1G70590,AT1G70600,AT1G70610,AT1G70620,AT1G70630,AT1G70640,AT1G70650,AT1G70660',
       'AT1G17500,AT1G17510,AT1G17520,AT1G17530,AT1G17540,AT1G17545,AT1G17550,AT1G17560,AT1G17580,AT1G17590,AT1G17600,AT1G17610,AT1G17615,AT1G17620,AT1G17630,AT1G17640,AT1G17650,AT1G17665,AT1G17680,AT1G17690,AT1G17700,AT1G17710,AT1G17720,AT1G17730,AT1G17744,AT1G17745,AT1G17750,AT1G17760,AT1G17770,AT1G17780,AT1G17790,AT1G17800,AT1G17810,AT1G17820,AT1G17830,AT1G17840,AT1G17850,AT1G17860,AT1G17870,AT1G17880,AT1G17890,AT1G17910,AT1G17920,AT1G17930,AT1G17940,AT1G17950,AT1G17960',
       'AT2G24230', 'AT2G27030',
       'AT2G17380,AT2G17390,AT2G17410,AT2G17420',
       'AT2G22475,AT2G22480,AT2G22490,AT2G22500,AT2G22510,AT2G22520,AT2G22530,AT2G22540,AT2G22560,AT2G22570,AT2G22590,AT2G22600,AT2G22610,AT2G22620,AT2G22630,AT2G22640,AT2G22650,AT2

In [32]:
# Get the values from 'matching_gene_id' column
matching_gene_ids = sign_wza_lfmm['matching_gene_id'].values

# Filter out None or NaN values
filtered_gene_ids = [gene_id for gene_id in matching_gene_ids if gene_id]

# Flatten the gene IDs (since some are comma-separated) into a single list
flattened_gene_ids = []
for gene_ids in filtered_gene_ids:
    flattened_gene_ids.extend(gene_ids.split(','))  # Split the comma-separated strings and extend the list

# Join all the gene IDs into a single string, separated by comma

In [45]:
len(flattened_gene_ids)

207

In [42]:
all_gene_ids_joined

'AT1G26680,AT1G30710,AT1G30720,AT1G55660,AT1G55670,AT1G55675,AT1G55680,AT1G55690,AT1G67560,AT1G67570,AT1G70020,AT1G11840,AT1G70580,AT1G70590,AT1G70600,AT1G70610,AT1G70620,AT1G70630,AT1G70640,AT1G70650,AT1G70660,AT1G17500,AT1G17510,AT1G17520,AT1G17530,AT1G17540,AT1G17545,AT1G17550,AT1G17560,AT1G17580,AT1G17590,AT1G17600,AT1G17610,AT1G17615,AT1G17620,AT1G17630,AT1G17640,AT1G17650,AT1G17665,AT1G17680,AT1G17690,AT1G17700,AT1G17710,AT1G17720,AT1G17730,AT1G17744,AT1G17745,AT1G17750,AT1G17760,AT1G17770,AT1G17780,AT1G17790,AT1G17800,AT1G17810,AT1G17820,AT1G17830,AT1G17840,AT1G17850,AT1G17860,AT1G17870,AT1G17880,AT1G17890,AT1G17910,AT1G17920,AT1G17930,AT1G17940,AT1G17950,AT1G17960,AT2G24230,AT2G27030,AT2G17380,AT2G17390,AT2G17410,AT2G17420,AT2G22475,AT2G22480,AT2G22490,AT2G22500,AT2G22510,AT2G22520,AT2G22530,AT2G22540,AT2G22560,AT2G22570,AT2G22590,AT2G22600,AT2G22610,AT2G22620,AT2G22630,AT2G22640,AT2G22650,AT2G22660,AT2G22670,AT2G22680,AT2G22690,AT2G22720,AT2G22730,AT2G22740,AT2G22750,AT2G22760

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Function to retrieve gene information from NCBI using the API
def get_gene_info_ncbi(gene_id):
    url = f"https://api.ncbi.nlm.nih.gov/datasets/v2alpha/gene/id/{gene_id}/download_summary"
    headers = {"accept": "application/json"}
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        # Return the JSON response
        return response.json()
    else:
        return None  # Return None if the request failed

# Example gene IDs (from your dataset)
gene_ids = ['59067', '7157']  # Replace these with the actual gene IDs from your dataset

# Fetch information for each gene
gene_info_list = []

for gene_id in gene_ids:
    gene_info = get_gene_info_ncbi(gene_id)
    if gene_info:
        # Extract relevant data from the response
        gene_data = {
            "gene_id": gene_id,
            "gene_name": gene_info['gene']['name'],
            "description": gene_info['gene']['description']
        }
        gene_info_list.append(gene_data)

# Convert the list of gene information to a DataFrame for easier handling
df_gene_info = pd.DataFrame(gene_info_list)

# Display the gene information DataFrame
print(df_gene_info)

# Now, assuming we have significant blocks from your data
# Let's integrate the gene information with the plot

# Significance threshold
threshold_value = 0.05 / len(wza_lfmm)
threshold = -np.log10(threshold_value)
biovar = 'bio1'

# Create chrom_pos in wza_lfmm by combining 'chrom' and 'pos'
wza_lfmm['chrom_pos'] = wza_lfmm['chrom'].astype(str) + '_' + wza_lfmm['pos'].astype(str)

# Copy the relevant columns for plotting
df = wza_lfmm[['Z_pVal', 'pos', 'chrom', 'chrom_pos']].copy()

# Parse chromosome number and position
df['chromosome'] = df['chrom']
df['position'] = df['pos']
df['-log10(pvalue)'] = -np.log10(df['Z_pVal'])

# Define colors for each chromosome or block
colors = sns.color_palette("crest", n_colors=len(df['chromosome'].unique()))  # Assign a unique color per chromosome

# Calculate chromosome offsets to prevent overlap
chromosome_offsets = {}
offset = 0
for chrom in sorted(df['chromosome'].unique()):
    chromosome_offsets[chrom] = offset
    max_position = df[df['chromosome'] == chrom]['position'].max()
    offset += max_position + 200  # Add buffer to prevent overlap

# Apply offsets to the position
df['adjusted_position'] = df.apply(lambda row: row['position'] + chromosome_offsets[row['chromosome']], axis=1)

# Create the Manhattan plot
plt.figure(figsize=(20, 6))

# Plot each chromosome separately
for chrom in sorted(df['chromosome'].unique()):
    subset = df[df['chromosome'] == chrom]
    plt.scatter(
        subset['adjusted_position'],
        subset


In [ ]:
1_1594

In [8]:
## all samples wza on kendalltau
wza_kendall = pd.read_csv('../wza/wza_kendalltau_results_bio1.csv')

In [9]:
sign_wza_kendall = wza_kendall[wza_kendall['Z_pVal'] <= 0.05/len(wza_kendall)]

In [10]:
sign_wza_kendall.sort_values('Z_pVal')

,gene,SNPs,hits,Z,top_candidate_p,Z_pVal
5406,2_1264,1885,0,2.057459,1.000000e+00,0.000000e+00
7217,2_973,1843,5,10.919003,9.999435e-01,0.000000e+00
10099,3_551,1849,4,5.318958,9.999891e-01,0.000000e+00
12569,4_2808,1729,220,20.956158,1.191537e-162,1.110223e-16
13384,4_801,225,146,31.615317,5.445274e-231,2.907737e-10
9133,3_2730,53,30,15.113784,4.984762e-46,2.548200e-09
7135,2_897,30,18,11.773300,7.715812e-29,4.589528e-08
431,1_1391,27,19,11.351576,2.056867e-32,6.736171e-08
3851,1_4506,109,8,17.976827,1.550062e-05,1.090242e-07
6199,2_199,10,9,9.224959,9.910000e-18,3.637041e-07


In [ ]:
kendall_sign = ['2_1264', '2_973' '3_551', '4_2808']

In [11]:
pd.merge(sign_wza_kendall,sign_wza_lfmm, on = 'gene')

,gene,SNPs_x,hits_x,Z_x,top_candidate_p_x,Z_pVal_x,SNPs_y,hits_y,Z_y,top_candidate_p_y,Z_pVal_y
0,2_1265,16,5,9.028052,3.984317e-07,2.119224e-06,16,15,11.151806,1.585000e-29,1.569795e-08
1,4_801,225,146,31.615317,5.445274e-231,2.907737e-10,225,130,23.960528,7.309601e-196,1.040595e-06


In [12]:
genes = ['2_1265', '4_801']

In [11]:
linages_wza_picmin = pd.read_csv('../linages_wza_picmin/linage_based_kendall_wza_picmin.csv')

In [21]:
linages_wza_picmin[linages_wza_picmin['n_est'] ==12].sort_values('p')

,index,numLin,p,q,n_est,locus,redundan,scaffold,start,chrom,pos
4172,4172,12,0.000024,0.001955,12,1_4847,1,4847,NaN,1,4847
12438,12438,12,0.000036,0.002633,12,4_2817,4,2817,NaN,4,2817
13736,13736,12,0.000337,0.015449,12,5_1272,5,1272,NaN,5,1272
5818,5818,12,0.000523,0.022242,12,2_170,2,170,NaN,2,170
259,259,12,0.000574,0.023699,12,1_1239,1,1239,NaN,1,1239
...,...,...,...,...,...,...,...,...,...,...,...
14707,14707,12,1.000000,1.000000,12,5_2166,5,2166,NaN,5,2166
12717,12717,12,1.000000,1.000000,12,4_325,4,325,NaN,4,325
9730,9730,12,1.000000,1.000000,12,3_3372,3,3372,NaN,3,3372
8996,8996,12,1.000000,1.000000,12,3_2698,3,2698,NaN,3,2698


In [12]:
th = 0.05/len(linages_wza_picmin)

In [13]:
sign_linages_wza_picmin = linages_wza_picmin[linages_wza_picmin['p'] <= th]

In [14]:
sign_linages_wza_picmin = sign_linages_wza_picmin.rename(columns = {'locus':'gene'})

In [17]:
sign_linages_wza_picmin.sort_values('n_est')

,index,numLin,p,q,n_est,gene,redundan,scaffold,start,chrom,pos
48,48,12,9.999990e-07,0.000158,2,1_1040,1,1040,NaN,1,1040
9684,9684,12,9.999990e-07,0.000158,2,3_3330,3,3330,NaN,3,3330
9683,9683,12,9.999990e-07,0.000158,2,3_333,3,333,NaN,3,333
9440,9440,12,9.999990e-07,0.000158,2,3_3104,3,3104,NaN,3,3104
9033,9033,12,9.999990e-07,0.000158,2,3_2730,3,2730,NaN,3,2730
...,...,...,...,...,...,...,...,...,...,...,...
14579,14579,12,9.999990e-07,0.000158,7,5_2048,5,2048,NaN,5,2048
3058,3058,12,9.999990e-07,0.000158,7,1_3821,1,3821,NaN,1,3821
3863,3863,12,9.999990e-07,0.000158,8,1_4562,1,4562,NaN,1,4562
15070,15070,12,9.999990e-07,0.000158,8,5_2505,5,2505,NaN,5,2505


In [ ]:
## 8 lineages adn significant 
lineages = ['4_1803', '5_2505', '1_4562' ]

# 12 lineages most significant 
lineages = ['1_4847']

In [15]:
sign_linages_wza_picmin.merge(sign_wza_kendall, on = 'gene')

,index,numLin,p,q,n_est,gene,redundan,scaffold,start,chrom,pos,SNPs,hits,Z,top_candidate_p,Z_pVal
0,48,12,9.999990e-07,0.000158,2,1_1040,1,1040,NaN,1,1040,93,37,14.712225,6.834746e-49,2.455139e-06
1,425,12,9.999990e-07,0.000158,3,1_1391,1,1391,NaN,1,1391,27,19,11.351576,2.056867e-32,6.736171e-08
2,3802,12,9.999990e-07,0.000158,2,1_4506,1,4506,NaN,1,4506,109,8,17.976827,1.550062e-05,1.090242e-07
3,5348,12,9.999990e-07,0.000158,5,2_1265,2,1265,NaN,2,1265,16,5,9.028052,3.984317e-07,2.119224e-06
4,6132,12,9.999990e-07,0.000158,6,2_199,2,199,NaN,2,199,10,9,9.224959,9.910000e-18,3.637041e-07
5,7058,12,9.999990e-07,0.000158,6,2_897,2,897,NaN,2,897,30,18,11.773300,7.715812e-29,4.589528e-08
6,9033,12,9.999990e-07,0.000158,2,3_2730,3,2730,NaN,3,2730,53,30,15.113784,4.984762e-46,2.548200e-09
7,9989,12,9.999990e-07,0.000158,2,3_551,3,551,NaN,3,551,1849,4,5.318958,9.999891e-01,0.000000e+00
8,12428,12,9.999990e-07,0.000158,2,4_2808,4,2808,NaN,4,2808,1729,220,20.956158,1.191537e-162,1.110223e-16
9,13229,12,9.999990e-07,0.000158,4,4_801,4,801,NaN,4,801,225,146,31.615317,5.445274e-231,2.907737e-10


In [23]:
genes = ['2_1265', '4_801']

In [28]:
sign_wza_lfmm.merge(sign_linages_wza_picmin, on = 'gene').merge(sign_wza_kendall, on = 'gene')

,gene,SNPs_x,hits_x,Z_x,top_candidate_p_x,Z_pVal_x,index,numLin,p,q,...,redundan,scaffold,start,chrom,pos,SNPs_y,hits_y,Z_y,top_candidate_p_y,Z_pVal_y
0,2_1265,16,15,11.151806,1.585000e-29,1.569795e-08,5348,12,9.999990e-07,0.000158,...,2,1265,NaN,2,1265,16,5,9.028052,3.984317e-07,2.119224e-06
1,4_801,225,130,23.960528,7.309601e-196,1.040595e-06,13229,12,9.999990e-07,0.000158,...,4,801,NaN,4,801,225,146,31.615317,5.445274e-231,2.907737e-10


In [30]:
gemma = pd.read_csv('/carnegie/nobackup/scratch/tbellagio/gea_grene-net/gwas/allele_assoc_runs/lmm_gemma/bio1/output/results_lmm.csv')

In [37]:
gemma_sign_blocks = gemma.groupby('blocks')['significant'].sum()[gemma.groupby('blocks')['significant'].sum() !=0].reset_index()

In [39]:
gemma_sign_blocks.merge(sign_wza_lfmm, left_on = 'blocks', right_on = 'gene')

,blocks,significant,gene,SNPs,hits,Z,top_candidate_p,Z_pVal


In [40]:
gemma_sign_blocks.merge(sign_linages_wza_picmin, left_on = 'blocks', right_on = 'gene')

,blocks,significant,index,numLin,p,q,n_est,gene,redundan,scaffold,start,chrom,pos


In [41]:
gemma_sign_blocks.merge(sign_wza_kendall, left_on = 'blocks', right_on = 'gene')

,blocks,significant,gene,SNPs,hits,Z,top_candidate_p,Z_pVal
0,2_973,4,2_973,1843,5,10.919003,0.999944,0.0


In [ ]:
#### addd to the annotated genes the blocks 

In [34]:
import pickle
dict_blocks = '../key_files/blocks_snpsid_dict.pkl'

with open(dict_blocks, 'rb') as file:
    dict_blocks = pickle.load(file)

reverse_mapping = {item: key for key, values in dict_blocks.items() for item in values}

In [35]:
gff3_formatted = pd.read_csv('../key_files/TAIR10_GFF3_genes_transposons_formatted4topr.csv')

In [37]:
from collections import defaultdict

In [60]:
# create clock ranges
blocks_ranges = {}
for key, values in dict_blocks.items():
    chromosome = dict_blocks[key][0].split('_')[0]
    start = dict_blocks[key][0].split('_')[1]
    end = dict_blocks[key][-1].split('_')[1]
    blocks_ranges[key] = [int(chromosome), int(start), int(end)]

In [70]:
# Step 2: Function to check if a gene falls within one or more blocks
def find_gene_blocks(chrom, gene_start, gene_end):
    matching_blocks = []
    for block, values in blocks_ranges.items():
        block_chrom, block_start, block_end = values
        if chrom == block_chrom:
            # Check if the gene overlaps with the block
            if (block_start <= gene_start <= block_end) or (block_start <= gene_end <= block_end) or (gene_start <= block_start <= gene_end):
                matching_blocks.append(block)
    return matching_blocks if matching_blocks else 'No matching block'

# Step 3: Apply the function to the dataframe

gff3_formatted['blocks'] = gff3_formatted.apply(lambda row: find_gene_blocks(row['chrom'], row['gene_start'], row['gene_end']), axis=1)


In [72]:
gff3_formatted.to_csv('../key_files/TAIR10_GFF3_genes_transposons_formatted4topr_w_blocks',index=None)

In [73]:
gff3_formatted

,Unnamed: 0,chrom,gene_start,gene_end,gene_symbol,biotype,gene_id,exon_chromstart,exon_chromend,blocks
0,0,1,3631,5899,AT1G01010,protein_coding_gene,AT1G01010,"3631,3996,4486,4706,5174,5439","3913,4276,4605,5095,5326,5899",[1_0]
1,6,1,5928,8737,AT1G01020,protein_coding_gene,AT1G01020,"8571,8417,8236,7942,7762,7564,7384,7157,6437,5...","8737,8464,8325,7987,7835,7649,7450,7232,7069,6...",[1_0]
2,24,1,11649,13714,AT1G01030,protein_coding_gene,AT1G01030,"13335,11649","13714,13173",[1_0]
3,26,1,23146,31227,AT1G01040,protein_coding_gene,AT1G01040,"23146,24542,24752,25041,25524,25825,26081,2629...","24451,24655,24962,25435,25743,25997,26203,2645...",[1_0]
4,66,1,31170,33153,AT1G01050,protein_coding_gene,AT1G01050,"33029,32547,32431,32282,32088,31933,31693,3152...","33153,32670,32459,32347,32195,31998,31813,3160...",[1_0]
...,...,...,...,...,...,...,...,...,...,...
27201,207189,5,26959569,26960323,AT5G67600,protein_coding_gene,AT5G67600,"26960181,26959937,26959569","26960323,26960093,26959799",[5_3109]
27202,207192,5,26960931,26963638,AT5G67610,protein_coding_gene,AT5G67610,"26963509,26963122,26962863,26962648,26962503,2...","26963638,26963335,26963020,26962793,26962562,2...",[5_3109]
27203,207212,5,26964770,26965996,AT5G67620,protein_coding_gene,AT5G67620,"26965468,26965198,26964770","26965996,26965379,26965004",[5_3109]
27204,207215,5,26967381,26969394,AT5G67630,protein_coding_gene,AT5G67630,"26968558,26967381","26969394,26968195",[5_3109]
